# RT-DETR-L — Query-based Object Detection (3-fold CV)

The RT-DETR-L pretrained model is fine-tuned on the competition's eight detection classes. The experiment uses the same video-grouped training frames as the YOLO and Faster R-CNN baselines: every sixth frame, three GroupKFold splits, batch size 2, and six epochs per fold.

Each fold starts from the same pretrained weights. The original `val/` videos are not used for cross-validation or training. Validation uses Ultralytics' built-in detector metrics; an identical COCO/ignored-region evaluation protocol is still needed for strict comparisons with the other architecture families.

**Run this notebook in a Kaggle GPU session with Internet enabled for the first download of `rtdetr-l.pt`.** The code does not create a Kaggle submission.

## 1 — Install and configure

In [1]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pyyaml', 'scikit-learn'],
    check=True,)

import gc
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.model_selection import GroupKFold
from ultralytics import RTDETR
import ultralytics

SEED = 42
FRAME_STEP = 6
N_FOLDS = 3
BATCH_SIZE = 2
EPOCHS = 6
IMAGE_SIZE = 640
MODEL_WEIGHTS = 'rtdetr-l.pt'

DATA_ROOT = Path('/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026')
WORK_ROOT = Path('/kaggle/working/rtdetr_3fold')
SPLIT_ROOT = WORK_ROOT / 'splits'
RUN_ROOT = WORK_ROOT / 'runs'
for folder in (WORK_ROOT, SPLIT_ROOT, RUN_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert DATA_ROOT.is_dir(), f'Missing competition dataset: {DATA_ROOT}'
assert (DATA_ROOT / 'train/annotations_coco.json').is_file()
assert torch.cuda.is_available(), 'Enable a GPU under Kaggle Notebook Settings'

print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Competition data:', DATA_ROOT)
print('Outputs:', WORK_ROOT)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
PyTorch: 2.11.0+cu130
Ultralytics: 8.4.203
GPU: Tesla T4
Competition data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026
Outputs: /kaggle/working/rtdetr_3fold


## 2 — Recreate the exact video-level sampling and folds

The procedure mirrors the previous YOLO notebook: sort frame filenames inside each video, take frames `[::6]`, then apply `GroupKFold(n_splits=3)` using the video clip as group. The fold counts are verified against the previous notebook. The original `val/` split remains untouched.

In [2]:
with open(DATA_ROOT / 'train/annotations_coco.json', 'r') as f:
    train_coco = json.load(f)

records = [
    {
        'image_id': int(im['id']),
        'file_name': str(im['file_name']),
        'clip': Path(im['file_name']).parent.name,
    }
    for im in train_coco['images']
]

frames = pd.DataFrame(records).sort_values(['clip', 'file_name']).reset_index(drop=True)
sampled = frames.loc[
    frames.groupby('clip', sort=True).cumcount() % FRAME_STEP == 0
].copy().reset_index(drop=True)

sampled['fold'] = -1
gkf = GroupKFold(n_splits=N_FOLDS)
for fold, (_, val_index) in enumerate(gkf.split(sampled, groups=sampled['clip'])):
    sampled.loc[val_index, 'fold'] = fold

assert len(frames) == 24201, f'Unexpected original frame count: {len(frames)}'
assert len(sampled) == 4062, f'Unexpected sampled frame count: {len(sampled)}'
assert sampled['fold'].between(0, 2).all()
expected_counts = {0: 1357, 1: 1356, 2: 1349}
actual_counts = sampled['fold'].value_counts().to_dict()
assert actual_counts == expected_counts, (actual_counts, expected_counts)

for fold in range(N_FOLDS):
    a = sampled.loc[sampled['fold'] != fold]
    b = sampled.loc[sampled['fold'] == fold]
    assert set(a['clip']).isdisjoint(set(b['clip']))
    assert set(a['image_id']).isdisjoint(set(b['image_id']))

sampled.to_csv(WORK_ROOT / 'cv_splits_step6.csv', index=False)
print('Original frames:', len(frames))
print('Sampled frames:', len(sampled))
print('Fold validation sizes:', actual_counts)
print('Video-level data leakage: none')

Original frames: 24201
Sampled frames: 4062
Fold validation sizes: {0: 1357, 1: 1356, 2: 1349}
Video-level data leakage: none


## 3 — Create Ultralytics-compatible data configurations

The original YOLO labels are reused without modifying the competition images. Crowd / ignored regions are not explicitly removed or modeled in this baseline. Image and label paths are checked before training.

In [6]:

import shutil
import yaml
from pathlib import Path
from tqdm.auto import tqdm

LOCAL_DATA = Path("/kaggle/working/rtdetr_local_data")
LOCAL_IMAGES = LOCAL_DATA / "images"
LOCAL_LABELS = LOCAL_DATA / "labels"

LOCAL_IMAGES.mkdir(parents=True, exist_ok=True)
LOCAL_LABELS.mkdir(parents=True, exist_ok=True)

assert len(sampled) == 4062
assert sampled["image_id"].is_unique

print("Preparing local dataset...")

for row in tqdm(
    sampled.itertuples(index=False),
    total=len(sampled),
    desc="Copying sampled frames"
):
    relative = Path(row.file_name)

    # Expected: images/<video_clip>/<frame>.jpg
    assert relative.parts[0] == "images"

    source_image = DATA_ROOT / "train" / relative
    source_label = (
        DATA_ROOT / "train" / "labels"
        / relative.parent.relative_to("images")
        / f"{relative.stem}.txt"
    )

    destination_image = LOCAL_DATA / relative
    destination_label = (
        LOCAL_LABELS
        / relative.parent.relative_to("images")
        / f"{relative.stem}.txt"
    )

    assert source_image.is_file(), source_image
    assert source_label.is_file(), source_label

    destination_image.parent.mkdir(
        parents=True, exist_ok=True
    )
    destination_label.parent.mkdir(
        parents=True, exist_ok=True
    )

    if (
        not destination_image.exists()
        or destination_image.stat().st_size
        != source_image.stat().st_size
    ):
        shutil.copy2(source_image, destination_image)

    if (
        not destination_label.exists()
        or destination_label.stat().st_size
        != source_label.stat().st_size
    ):
        shutil.copy2(source_label, destination_label)

print("Local dataset prepared.")

# Create train/validation lists for each fold
LOCAL_SPLITS = Path(
    "/kaggle/working/rtdetr_local_splits"
)
LOCAL_SPLITS.mkdir(parents=True, exist_ok=True)

fold_yamls = {}

for fold in range(3):
    fold_dir = LOCAL_SPLITS / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_part = sampled[sampled["fold"] != fold]
    val_part = sampled[sampled["fold"] == fold]

    def paths_for(part):
        return [
            str(LOCAL_DATA / Path(name))
            for name in part["file_name"]
        ]

    train_paths = paths_for(train_part)
    val_paths = paths_for(val_part)

    assert len(train_paths) + len(val_paths) == 4062

    (fold_dir / "train.txt").write_text(
        "\n".join(train_paths) + "\n"
    )
    (fold_dir / "val.txt").write_text(
        "\n".join(val_paths) + "\n"
    )

    config = {
        "train": str(fold_dir / "train.txt"),
        "val": str(fold_dir / "val.txt"),
        "nc": 8,
        "names": CLASS_NAMES,
    }

    yaml_path = fold_dir / "data.yaml"

    with yaml_path.open("w") as f:
        yaml.safe_dump(config, f, sort_keys=False)

    fold_yamls[fold] = yaml_path

    print(
        f"Fold {fold}: "
        f"train={len(train_paths)}, "
        f"val={len(val_paths)}"
    )

print("All local fold configurations are ready.")

Preparing local dataset...


Copying sampled frames:   0%|          | 0/4062 [00:00<?, ?it/s]

Local dataset prepared.
Fold 0: train=2705, val=1357
Fold 1: train=2706, val=1356
Fold 2: train=2713, val=1349
All local fold configurations are ready.


## 4 — Prepare pretrained RT-DETR and record the experiment configuration

`rtdetr-l.pt` is a pretrained large query-based detector. RT-DETR has a different cost profile from YOLO11n: batch size 2 and 640px resolution may require much more GPU time and memory. The first experiment uses AdamW, with parameters held constant across folds. `deterministic=False` avoids unsupported deterministic CUDA backward paths in deformable attention. AMP is disabled to reduce risk of non-finite matching losses.

In [4]:
experiment_config = {
    'architecture': 'RT-DETR-L',
    'weights': MODEL_WEIGHTS,
    'frame_step': FRAME_STEP,
    'n_folds': N_FOLDS,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'image_size': IMAGE_SIZE,
    'seed': SEED,
    'optimizer': 'AdamW',
    'lr0': 1e-4,
    'weight_decay': 1e-4,
    'amp': False,
    'deterministic': False,
    'num_workers': 0,
    'mosaic': 0.0,
    'validation': 'Ultralytics detection mAP50-95, mAP50, mAP75 (not ignore-aware COCOeval)',
}
(WORK_ROOT / 'config.json').write_text(json.dumps(experiment_config, indent=2))

# Download COCO-pretrained RT-DETR-L weights once. Internet must be on.
probe_model = RTDETR(MODEL_WEIGHTS)
assert len(probe_model.names) > 0
print('Pretrained RT-DETR weights loaded.')
print('Pretraining classes:', len(probe_model.names))
del probe_model
gc.collect()
torch.cuda.empty_cache()

Pretrained RT-DETR weights loaded.
Pretraining classes: 80


## 5 — Train all three folds, then evaluate each fold

The model is recreated from `rtdetr-l.pt` for every fold. Results are written after each **completed** fold, and model weights are saved inside `/kaggle/working`. A finished Kaggle notebook version preserves these files in its Output. Kaggle does not guarantee recovery of files from an interrupted/failed commit, so monitor the first fold before running an unattended commit.

The next fold will not start when the notebook has already used nine hours, leaving a margin under the normal Kaggle 12-hour commit limit. This is not a hard timeout: a running fold can still exceed the limit.

In [8]:

import gc
import json
import time
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from ultralytics import RTDETR

NEW_RUN_ROOT = Path(
    "/kaggle/working/rtdetr_separate_folds"
)
NEW_RUN_ROOT.mkdir(parents=True, exist_ok=True)

def train_rtdetr_fold(fold):

    assert fold in (0, 1, 2)
    assert torch.cuda.is_available()
    assert fold_yamls[fold].is_file()

    run_name = f"fold_{fold}"
    run_dir = NEW_RUN_ROOT / run_name

    metrics_path = run_dir / "fold_metrics.csv"
    best_path = run_dir / "weights" / "best.pt"
    last_path = run_dir / "weights" / "last.pt"

    # If already fully completed, do not train twice
    if metrics_path.exists() and best_path.exists():
        print(f"Fold {fold} already completed.")
        return pd.read_csv(metrics_path)

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)

    gc.collect()
    torch.cuda.empty_cache()

    print("=" * 55)
    print(f"STARTING RT-DETR-L — FOLD {fold}")
    print("=" * 55)

    start = time.monotonic()

    # Resume if an unfinished checkpoint exists
    if last_path.exists():
        print(f"Resuming fold {fold} from last.pt")
        model = RTDETR(str(last_path))
        model.train(
            resume=True,
            device=0,
            workers=0,
        )
    else:
        model = RTDETR("rtdetr-l.pt")

        model.train(
            data=str(fold_yamls[fold]),
            epochs=6,
            imgsz=640,
            batch=2,
            nbs=2,
            device=0,
            workers=0,
            optimizer="AdamW",
            lr0=0.0001,
            lrf=0.1,
            weight_decay=0.0001,
            warmup_epochs=1.0,
            cos_lr=True,
            mosaic=0.0,
            amp=False,
            deterministic=False,
            seed=42,
            cache=False,
            patience=6,
            save=True,
            save_period=1,
            plots=False,
            project=str(NEW_RUN_ROOT),
            name=run_name,
            exist_ok=True,
        )

    train_minutes = (
        time.monotonic() - start
    ) / 60

    assert best_path.exists(), (
        f"Best checkpoint missing: {best_path}"
    )

    print(f"Evaluating fold {fold}...")

    best_model = RTDETR(str(best_path))

    metrics = best_model.val(
        data=str(fold_yamls[fold]),
        split="val",
        imgsz=640,
        batch=2,
        device=0,
        workers=0,
        plots=False,
        project=str(NEW_RUN_ROOT),
        name=f"fold_{fold}_evaluation",
        exist_ok=True,
    )

    result = {
        "fold": fold,
        "model": "RT-DETR-L",
        "epochs": 6,
        "imgsz": 640,
        "batch": 2,
        "optimizer": "AdamW",
        "mAP50-95": float(metrics.box.map),
        "mAP50": float(metrics.box.map50),
        "mAP75": float(metrics.box.map75),
        "train_minutes": round(train_minutes, 2),
        "checkpoint": str(best_path),
    }

    result_df = pd.DataFrame([result])
    result_df.to_csv(metrics_path, index=False)

    print(f"\nFOLD {fold} COMPLETED")
    print(result_df.to_string(index=False))

    del model, best_model, metrics
    gc.collect()
    torch.cuda.empty_cache()

    return result_df

print("Single-fold training function is ready.")


Single-fold training function is ready.


In [9]:

FOLD_TO_RUN = 0

result = train_rtdetr_fold(FOLD_TO_RUN)
display(result)

STARTING RT-DETR-L — FOLD 0
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_local_splits/fold_0/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fold_0, 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


        2/6      3.69G      0.795     0.6458     0.1051         46        640: 100% ━━━━━━━━━━━━ 1353/1353 2.2it/s 10:060.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 3.1it/s 1:490.3sss
                   all       1357      59035      0.291       0.31      0.221      0.112

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        3/6      3.69G      0.741     0.5611    0.08977         88        640: 100% ━━━━━━━━━━━━ 1353/1353 2.2it/s 10:060.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 340/340 3.1it/s 1:490.3sss
                   all       1357      59035      0.323      0.322      0.255      0.127

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
        4/6      3.69G     0.7034     0.5249    0.08284         27        640: 100% ━━━━━━━━━━━━ 1353/1353 2.2it/s 10:060.4s
                 Cl

,fold,model,epochs,imgsz,batch,optimizer,mAP50-95,mAP50,mAP75,train_minutes,checkpoint
0,0,RT-DETR-L,6,640,2,AdamW,0.155325,0.312188,0.137966,73.31,/kaggle/working/rtdetr_separate_folds/fold_0/w...


In [15]:

from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import pandas as pd

RUN_ROOT = Path("/kaggle/working/rtdetr_separate_folds")
BACKUP_DIR = Path("/kaggle/working/rtdetr_backups")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def backup_fold_compact(fold):
    fold_dir = RUN_ROOT / f"fold_{fold}"

    files_to_save = [
        "weights/best.pt",
        "weights/last.pt",
        "fold_metrics.csv",
        "results.csv",
        "args.yaml",
    ]

    archive_path = (
        BACKUP_DIR / f"rtdetr_fold_{fold}_compact.zip"
    )

    with ZipFile(
        archive_path,
        "w",
        compression=ZIP_DEFLATED,
        compresslevel=1
    ) as archive:

        for relative in files_to_save:
            path = fold_dir / relative

            if path.exists():
                archive.write(
                    path,
                    arcname=f"fold_{fold}/{relative}"
                )
            else:
                print("Optional file not found:", relative)

    with ZipFile(archive_path) as archive:
        assert archive.testzip() is None

    print("Backup created:", archive_path)
    print(
        "Size:",
        round(archive_path.stat().st_size / 1e6, 2),
        "MB"
    )

    return archive_path




In [18]:

backup_fold_compact(0)


Backup created: /kaggle/working/rtdetr_backups/rtdetr_fold_0_compact.zip
Size: 122.37 MB


PosixPath('/kaggle/working/rtdetr_backups/rtdetr_fold_0_compact.zip')

In [19]:
FOLD_TO_RUN = 1

result = train_rtdetr_fold(FOLD_TO_RUN)
display(result)

STARTING RT-DETR-L — FOLD 1
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_local_splits/fold_1/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fold_1, 

,fold,model,epochs,imgsz,batch,optimizer,mAP50-95,mAP50,mAP75,train_minutes,checkpoint
0,1,RT-DETR-L,6,640,2,AdamW,0.153484,0.298599,0.137065,73.17,/kaggle/working/rtdetr_separate_folds/fold_1/w...


In [20]:

backup_fold_compact(1)


Backup created: /kaggle/working/rtdetr_backups/rtdetr_fold_1_compact.zip
Size: 122.37 MB


PosixPath('/kaggle/working/rtdetr_backups/rtdetr_fold_1_compact.zip')

In [21]:
FOLD_TO_RUN = 2

result = train_rtdetr_fold(FOLD_TO_RUN)
display(result)

STARTING RT-DETR-L — FOLD 2
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/rtdetr_local_splits/fold_2/data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=fold_2, 

,fold,model,epochs,imgsz,batch,optimizer,mAP50-95,mAP50,mAP75,train_minutes,checkpoint
0,2,RT-DETR-L,6,640,2,AdamW,0.14209,0.275395,0.133256,73.41,/kaggle/working/rtdetr_separate_folds/fold_2/w...


In [22]:

backup_fold_compact(2)


Backup created: /kaggle/working/rtdetr_backups/rtdetr_fold_2_compact.zip
Size: 122.37 MB


PosixPath('/kaggle/working/rtdetr_backups/rtdetr_fold_2_compact.zip')

## 6 — Check output files

This final cell confirms which folds have metrics and saved weights. It does not create a competition submission.

In [23]:

results = []

for fold in range(3):
    path = (
        NEW_RUN_ROOT
        / f"fold_{fold}"
        / "fold_metrics.csv"
    )

    if path.exists():
        results.append(pd.read_csv(path))
    else:
        print(f"Fold {fold}: not completed in this runtime")

if results:
    cv_results = pd.concat(
        results,
        ignore_index=True
    ).sort_values("fold")

    display(cv_results)

    cv_results.to_csv(
        NEW_RUN_ROOT / "cv_results.csv",
        index=False
    )

    if len(cv_results) == 3:
        summary = cv_results[
            ["mAP50-95", "mAP50", "mAP75"]
        ].agg(["mean", "std"])

        display(summary)

        summary.to_csv(
            NEW_RUN_ROOT / "cv_summary.csv"
        )
        print("Full 3-fold CV completed.")
    else:
        print(
            f"Available results: {len(cv_results)}/3"
        )


,fold,model,epochs,imgsz,batch,optimizer,mAP50-95,mAP50,mAP75,train_minutes,checkpoint
0,0,RT-DETR-L,6,640,2,AdamW,0.155325,0.312188,0.137966,73.31,/kaggle/working/rtdetr_separate_folds/fold_0/w...
1,1,RT-DETR-L,6,640,2,AdamW,0.153484,0.298599,0.137065,73.17,/kaggle/working/rtdetr_separate_folds/fold_1/w...
2,2,RT-DETR-L,6,640,2,AdamW,0.142090,0.275395,0.133256,73.41,/kaggle/working/rtdetr_separate_folds/fold_2/w...


,mAP50-95,mAP50,mAP75
mean,0.150299,0.295394,0.136096
std,0.007169,0.018605,0.002500


Full 3-fold CV completed.
